###### 131_4_practical_exclude_old_parts.ipynb

In [1]:
import pandas as pd

In [2]:
df_defect = pd.read_csv("dataset/defect_old_parts.csv")
df_defect

,defect_id,site,model,occ_month,defect_category,defect_qty,comment
0,D001,SUS,J69P,2026-04,Buck-IC,3,Buck-ICのHS-FET損傷を確認
1,D002,SUS,J69P,2026-04,Coil,1,旧部品OLD-110で発生した過去データ
2,D003,WSE,310D,2026-05,Coil,2,コイル内部に鉄粉噛み込みを確認
3,D004,GSE,310D,2026-05,NTF,1,市場返却品を確認したが不具合再現せず
4,D005,MELMB,J69P,2026-06,Buck-IC,4,OLD-220使用時の参考登録データ
5,D006,WSE,310D,2026-06,Coil,3,160Vインパルス検査で異常検出
6,D007,SUS,J69P,2026-06,NTF,1,現品確認の結果、症状再現せず
7,D008,GSE,310D,2026-06,Coil,2,旧仕様OLD-110からの切替前データ
8,D009,MELMB,310D,2026-07,Buck-IC,5,Buck-IC再発疑いとして調査中
9,D010,WSE,J69P,2026-07,Coil,1,OLD-220評価時のメモ


In [3]:
exclude_pattern = "OLD-110|OLD-220"
exclude_pattern

'OLD-110|OLD-220'

In [4]:
df_defect["exclude_flg"] = df_defect["comment"].str.contains(
    exclude_pattern,
    case=False,
    na=False,
)
df_defect

,defect_id,site,model,occ_month,defect_category,defect_qty,comment,exclude_flg
0,D001,SUS,J69P,2026-04,Buck-IC,3,Buck-ICのHS-FET損傷を確認,False
1,D002,SUS,J69P,2026-04,Coil,1,旧部品OLD-110で発生した過去データ,True
2,D003,WSE,310D,2026-05,Coil,2,コイル内部に鉄粉噛み込みを確認,False
3,D004,GSE,310D,2026-05,NTF,1,市場返却品を確認したが不具合再現せず,False
4,D005,MELMB,J69P,2026-06,Buck-IC,4,OLD-220使用時の参考登録データ,True
5,D006,WSE,310D,2026-06,Coil,3,160Vインパルス検査で異常検出,False
6,D007,SUS,J69P,2026-06,NTF,1,現品確認の結果、症状再現せず,False
7,D008,GSE,310D,2026-06,Coil,2,旧仕様OLD-110からの切替前データ,True
8,D009,MELMB,310D,2026-07,Buck-IC,5,Buck-IC再発疑いとして調査中,False
9,D010,WSE,J69P,2026-07,Coil,1,OLD-220評価時のメモ,True


In [5]:
df_actual = df_defect[
    ~df_defect["exclude_flg"]
].copy()
df_actual

,defect_id,site,model,occ_month,defect_category,defect_qty,comment,exclude_flg
0,D001,SUS,J69P,2026-04,Buck-IC,3,Buck-ICのHS-FET損傷を確認,False
2,D003,WSE,310D,2026-05,Coil,2,コイル内部に鉄粉噛み込みを確認,False
3,D004,GSE,310D,2026-05,NTF,1,市場返却品を確認したが不具合再現せず,False
5,D006,WSE,310D,2026-06,Coil,3,160Vインパルス検査で異常検出,False
6,D007,SUS,J69P,2026-06,NTF,1,現品確認の結果、症状再現せず,False
8,D009,MELMB,310D,2026-07,Buck-IC,5,Buck-IC再発疑いとして調査中,False
10,D011,SUS,310D,2026-07,Coil,2,コイル端子部に変色あり,False
11,D012,MELMB,J69P,2026-07,NTF,1,ユーザー申告のみで現象確認できず,False


In [8]:
df_exclude = df_defect[
    df_defect["exclude_flg"]
].copy()
df_exclude

,defect_id,site,model,occ_month,defect_category,defect_qty,comment,exclude_flg
1,D002,SUS,J69P,2026-04,Coil,1,旧部品OLD-110で発生した過去データ,True
4,D005,MELMB,J69P,2026-06,Buck-IC,4,OLD-220使用時の参考登録データ,True
7,D008,GSE,310D,2026-06,Coil,2,旧仕様OLD-110からの切替前データ,True
9,D010,WSE,J69P,2026-07,Coil,1,OLD-220評価時のメモ,True


In [9]:
df_site_count = df_actual.groupby(
    "site",
    as_index=False,
)["defect_id"].count()
df_site_count

,site,defect_id
0,GSE,1
1,MELMB,2
2,SUS,3
3,WSE,2


In [10]:
df_site_count = df_site_count.rename(
    columns={
        "defect_id": "actual_count",
    }
)
df_site_count

,site,actual_count
0,GSE,1
1,MELMB,2
2,SUS,3
3,WSE,2


In [11]:
df_site_count = df_site_count.sort_values(
    ["actual_count", "site"],
    ascending=[False, True],
).reset_index(drop=True)
df_site_count

,site,actual_count
0,SUS,3
1,MELMB,2
2,WSE,2
3,GSE,1


In [12]:
df_site_qty = df_actual.groupby(
    "site",
    as_index=False,
)["defect_qty"].sum()
df_site_qty

,site,defect_qty
0,GSE,1
1,MELMB,6
2,SUS,6
3,WSE,5


In [13]:
df_site_qty = df_site_qty.rename(
    columns={
        "defect_qty": "actual_defect_qty",
    }
)
df_site_qty

,site,actual_defect_qty
0,GSE,1
1,MELMB,6
2,SUS,6
3,WSE,5


In [15]:
df_site_qty = df_site_qty.sort_values(
    ["actual_defect_qty", "site"],
    ascending=[False, True],
).reset_index(drop=True)
df_site_qty

,site,actual_defect_qty
0,MELMB,6
1,SUS,6
2,WSE,5
3,GSE,1
